# Part 3 — Churn Prediction Model & Model Card
**D2C Customer Churn Intelligence Capstone**

**Snapshot date:** `2025-09-30` | **Target:** `churn_next_60d` | **Split:** pre-assigned train/validation/test

---
### Notebook Structure
1. Setup & Data Loading
2. Feature Preparation & Leakage Check
3. Train/Validation/Test Split
4. Baseline Model (Logistic Regression)
5. Stronger Model (XGBoost)
6. Evaluation & Comparison
7. Threshold Selection & Business Justification
8. Feature Importance / Interpretability
9. Error Analysis (10 customer examples)
10. Save Model & Export Metrics

## 1. Setup & Data Loading

In [ ]:
!pip install -q xgboost

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json, joblib, warnings
from IPython.display import display
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (classification_report, confusion_matrix,
                              roc_auc_score, precision_recall_curve,
                              roc_curve, f1_score, average_precision_score)
from xgboost import XGBClassifier

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 130, 'figure.figsize': (10, 4)})

SNAPSHOT = pd.Timestamp('2025-09-30')
print('Libraries loaded ✓')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/d2c_churn_data/'   # <── update if needed

# Use the pre-built modeling snapshot (leakage-free per data dictionary)
df = pd.read_csv(DATA_DIR + 'rfm_modeling_snapshot.csv')
print(f'Modeling snapshot: {df.shape[0]:,} rows × {df.shape[1]} cols')
display(df.head(3))

## 2. Feature Preparation & Leakage Check

In [ ]:
# ── Identify columns ──────────────────────────────────────────────────────────
target_col = 'churn_next_60d'
id_cols    = ['customer_id', 'snapshot_date', 'split']
leak_cols  = [target_col]  # must NOT be a feature

feature_cols = [c for c in df.columns if c not in id_cols + leak_cols]
print(f'Features ({len(feature_cols)}):')
print(feature_cols)

In [ ]:
# ── Leakage verification ──────────────────────────────────────────────────────
print('LEAKAGE CHECK:')
print(f'  Target column ({target_col}): EXCLUDED from features ✓')
print(f'  snapshot_date: EXCLUDED (identifier, not feature) ✓')
print(f'  split column: EXCLUDED (used only for splitting) ✓')
print(f'  All features in rfm_modeling_snapshot are pre-built from pre-snapshot data (per data dictionary) ✓')
print(f'\n  No post-snapshot data leaks into features.')

In [ ]:
# ── Encode categorical features ──────────────────────────────────────────────
cat_cols = df[feature_cols].select_dtypes(include='object').columns.tolist()
num_cols = [c for c in feature_cols if c not in cat_cols]

print(f'Categorical: {cat_cols}')
print(f'Numerical:   {len(num_cols)} columns')

# One-hot encode categoricals
df_encoded = pd.get_dummies(df, columns=cat_cols, drop_first=True, dummy_na=True)

# Update feature cols after encoding
feature_cols_encoded = [c for c in df_encoded.columns if c not in id_cols + leak_cols]
print(f'\nEncoded features: {len(feature_cols_encoded)}')

In [ ]:
# ── Handle missing values ─────────────────────────────────────────────────────
missing = df_encoded[feature_cols_encoded].isnull().sum()
missing = missing[missing > 0]
if len(missing):
    print('Missing values in features:')
    print(missing)
    # Fill numerical NaNs with median
    for col in missing.index:
        df_encoded[col] = df_encoded[col].fillna(df_encoded[col].median())
    print('\nFilled with median.')
else:
    print('No missing values in encoded features ✓')

## 3. Train / Validation / Test Split

In [ ]:
# ── Use the pre-assigned split ────────────────────────────────────────────────
train_df = df_encoded[df_encoded['split'] == 'train']
val_df   = df_encoded[df_encoded['split'] == 'validation']
test_df  = df_encoded[df_encoded['split'] == 'test']

X_train, y_train = train_df[feature_cols_encoded], train_df[target_col]
X_val,   y_val   = val_df[feature_cols_encoded],   val_df[target_col]
X_test,  y_test  = test_df[feature_cols_encoded],  test_df[target_col]

print(f'Train:      {X_train.shape[0]:,} rows | Churn rate: {y_train.mean()*100:.1f}%')
print(f'Validation: {X_val.shape[0]:,} rows | Churn rate: {y_val.mean()*100:.1f}%')
print(f'Test:       {X_test.shape[0]:,} rows | Churn rate: {y_test.mean()*100:.1f}%')

In [ ]:
# ── Scale features for Logistic Regression ────────────────────────────────────
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_val_sc   = scaler.transform(X_val)
X_test_sc  = scaler.transform(X_test)

## 4. Baseline Model — Logistic Regression

In [ ]:
# ── Train baseline ────────────────────────────────────────────────────────────
lr = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
lr.fit(X_train_sc, y_train)

lr_probs_val  = lr.predict_proba(X_val_sc)[:, 1]
lr_preds_val  = (lr_probs_val >= 0.5).astype(int)

print('=== BASELINE: Logistic Regression (Validation) ===')
print(classification_report(y_val, lr_preds_val, target_names=['No Churn','Churned']))
print(f'ROC-AUC: {roc_auc_score(y_val, lr_probs_val):.4f}')
print(f'PR-AUC:  {average_precision_score(y_val, lr_probs_val):.4f}')

## 5. Stronger Model — XGBoost

In [ ]:
# ── Train XGBoost ─────────────────────────────────────────────────────────────
# Calculate scale_pos_weight for class imbalance
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
spw = neg_count / pos_count
print(f'Class balance — No Churn: {neg_count}, Churned: {pos_count}, scale_pos_weight: {spw:.2f}')

xgb = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=spw,
    eval_metric='logloss',
    random_state=42,
    use_label_encoder=False,
)

xgb.fit(X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=50)

xgb_probs_val = xgb.predict_proba(X_val)[:, 1]
xgb_preds_val = (xgb_probs_val >= 0.5).astype(int)

print('\n=== XGBoost (Validation, threshold=0.5) ===')
print(classification_report(y_val, xgb_preds_val, target_names=['No Churn','Churned']))
print(f'ROC-AUC: {roc_auc_score(y_val, xgb_probs_val):.4f}')
print(f'PR-AUC:  {average_precision_score(y_val, xgb_probs_val):.4f}')

## 6. Evaluation & Comparison

In [ ]:
# ── ROC and PR curves ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC curve
for name, probs, color in [('Logistic Regression', lr_probs_val, 'blue'),
                            ('XGBoost', xgb_probs_val, 'red')]:
    fpr, tpr, _ = roc_curve(y_val, probs)
    auc = roc_auc_score(y_val, probs)
    axes[0].plot(fpr, tpr, color=color, label=f'{name} (AUC={auc:.3f})')
axes[0].plot([0,1],[0,1],'k--', alpha=0.3)
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve')
axes[0].legend()

# PR curve
for name, probs, color in [('Logistic Regression', lr_probs_val, 'blue'),
                            ('XGBoost', xgb_probs_val, 'red')]:
    prec, rec, _ = precision_recall_curve(y_val, probs)
    ap = average_precision_score(y_val, probs)
    axes[1].plot(rec, prec, color=color, label=f'{name} (AP={ap:.3f})')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve')
axes[1].legend()

plt.suptitle('Model Comparison — Validation Set', fontweight='bold')
plt.tight_layout()
plt.savefig('chart_p3_01_roc_pr_curves.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Confusion matrices side by side ──────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, name, preds in [(axes[0], 'Logistic Regression', lr_preds_val),
                         (axes[1], 'XGBoost', xgb_preds_val)]:
    cm = confusion_matrix(y_val, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['No Churn','Churned'], yticklabels=['No Churn','Churned'])
    ax.set_title(f'{name} (threshold=0.5)')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.suptitle('Confusion Matrices — Validation Set', fontweight='bold')
plt.tight_layout()
plt.savefig('chart_p3_02_confusion_matrices.png', bbox_inches='tight')
plt.show()

## 7. Threshold Selection & Business Justification

In [ ]:
# ── Business-driven threshold selection ───────────────────────────────────────
# In churn prediction:
#   False Negative (missed churner) = customer leaves → high cost (lost LTV)
#   False Positive (false alarm)     = we spend on retention for someone who'd stay → lower cost
#
# Therefore we want HIGH RECALL (catch churners) even at cost of some precision.
# We'll lower the threshold below 0.5 to favour recall.

thresholds = np.arange(0.25, 0.65, 0.05)
results = []

for t in thresholds:
    preds = (xgb_probs_val >= t).astype(int)
    cm = confusion_matrix(y_val, preds)
    tn, fp, fn, tp = cm.ravel()
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0
    rec  = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0
    results.append({'threshold': t, 'precision': prec, 'recall': rec,
                    'f1': f1, 'TP': tp, 'FP': fp, 'FN': fn, 'TN': tn})

thresh_df = pd.DataFrame(results)
display(thresh_df.round(3))

In [ ]:
# ── Visualise threshold trade-off ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(thresh_df['threshold'], thresh_df['precision'], 'b-o', label='Precision')
ax.plot(thresh_df['threshold'], thresh_df['recall'], 'r-o', label='Recall')
ax.plot(thresh_df['threshold'], thresh_df['f1'], 'g-o', label='F1 Score')
ax.set_xlabel('Threshold')
ax.set_ylabel('Score')
ax.set_title('Threshold vs Precision / Recall / F1 (XGBoost, Validation)')
ax.legend()
ax.grid(True, alpha=0.3)

# Mark selected threshold
best_f1_idx = thresh_df['f1'].idxmax()
best_thresh = thresh_df.loc[best_f1_idx, 'threshold']
ax.axvline(best_thresh, color='green', linestyle='--', alpha=0.5, label=f'Best F1 @ {best_thresh:.2f}')

# Also mark a recall-focused threshold
recall_target = thresh_df[thresh_df['recall'] >= 0.70]
if len(recall_target):
    recall_thresh = recall_target.loc[recall_target['f1'].idxmax(), 'threshold']
    ax.axvline(recall_thresh, color='red', linestyle='--', alpha=0.5, label=f'Recall≥70% @ {recall_thresh:.2f}')
else:
    recall_thresh = best_thresh

ax.legend()
plt.tight_layout()
plt.savefig('chart_p3_03_threshold_tradeoff.png', bbox_inches='tight')
plt.show()

# Select threshold: prioritise recall ≥ 70% with best F1
SELECTED_THRESHOLD = recall_thresh
print(f'\n=== SELECTED THRESHOLD: {SELECTED_THRESHOLD:.2f} ===')
print(f'Rationale: In churn prediction, missing a churner (FN) costs the business more')
print(f'than a false alarm (FP). A false alarm costs ~₹30-40 (campaign cost) while a')
print(f'missed churner costs ₹500-5000+ in lost LTV. We lower the threshold to catch')
print(f'more churners, accepting some false positives.')

selected_row = thresh_df[thresh_df['threshold'] == SELECTED_THRESHOLD].iloc[0]
print(f'\nAt threshold {SELECTED_THRESHOLD:.2f}:')
print(f'  Precision: {selected_row["precision"]:.3f}')
print(f'  Recall:    {selected_row["recall"]:.3f}')
print(f'  F1:        {selected_row["f1"]:.3f}')

## 8. Feature Importance / Interpretability

In [ ]:
# ── XGBoost feature importance ────────────────────────────────────────────────
importance = pd.Series(xgb.feature_importances_, index=feature_cols_encoded)
top_20 = importance.nlargest(20)

fig, ax = plt.subplots(figsize=(10, 7))
top_20.sort_values().plot(kind='barh', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Top 20 Feature Importances (XGBoost gain)')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.savefig('chart_p3_04_feature_importance.png', bbox_inches='tight')
plt.show()

print('Top 10 features driving churn prediction:')
for i, (feat, imp) in enumerate(importance.nlargest(10).items(), 1):
    print(f'  {i:>2}. {feat:<35} importance={imp:.4f}')

## 9. Error Analysis (10+ Customer Examples)

In [ ]:
# ── Apply selected threshold to validation set ────────────────────────────────
val_analysis = val_df[['customer_id']].copy()
val_analysis['actual']      = y_val.values
val_analysis['prob']        = xgb_probs_val
val_analysis['predicted']   = (xgb_probs_val >= SELECTED_THRESHOLD).astype(int)

# Merge key features for interpretation
key_feats = ['customer_id','recency_days','frequency_180d','monetary_180d',
             'ticket_count_90d','sessions_30d','return_rate_180d','last_visit_days_ago']
key_feats_avail = [c for c in key_feats if c in df.columns]
val_analysis = val_analysis.merge(df[key_feats_avail], on='customer_id', how='left')

# Classify error types
val_analysis['error_type'] = 'Correct'
val_analysis.loc[(val_analysis['actual']==1) & (val_analysis['predicted']==0), 'error_type'] = 'False Negative'
val_analysis.loc[(val_analysis['actual']==0) & (val_analysis['predicted']==1), 'error_type'] = 'False Positive'

print('Error distribution:')
print(val_analysis['error_type'].value_counts())

In [ ]:
# ── Select 5 False Negatives and 5 False Positives ───────────────────────────
fn_cases = val_analysis[val_analysis['error_type'] == 'False Negative'].head(5)
fp_cases = val_analysis[val_analysis['error_type'] == 'False Positive'].head(5)

error_cases = pd.concat([fn_cases, fp_cases])

print('\n=== FALSE NEGATIVES (Model missed these churners) ===')
print('Business risk: These customers will churn but the model says they are safe.')
print('They will NOT receive any retention outreach and will be lost.\n')
display(fn_cases.round(3))

print('\n=== FALSE POSITIVES (Model flagged these but they did NOT churn) ===')
print('Business risk: Wasted campaign budget (~₹30-40 per customer) on retention')
print('for someone who would have stayed anyway.\n')
display(fp_cases.round(3))

In [ ]:
# ── Detailed interpretation of each error case ────────────────────────────────
print('\n=== DETAILED ERROR INTERPRETATION ===')

for i, (_, row) in enumerate(error_cases.iterrows(), 1):
    cid = row['customer_id']
    etype = row['error_type']
    prob = row['prob']
    
    print(f'\n--- Case {i}: {cid} ({etype}) ---')
    print(f'  Predicted prob: {prob:.3f} | Threshold: {SELECTED_THRESHOLD:.2f} | Actual: {"Churned" if row["actual"] else "Retained"}')
    
    feat_str = []
    for feat in ['recency_days','frequency_180d','monetary_180d','ticket_count_90d','sessions_30d','last_visit_days_ago']:
        if feat in row.index and pd.notna(row[feat]):
            feat_str.append(f'{feat}={row[feat]:.0f}')
    print(f'  Features: {", ".join(feat_str)}')
    
    if etype == 'False Negative':
        print(f'  WHY MISSED: Probability ({prob:.3f}) is below threshold ({SELECTED_THRESHOLD:.2f}).')
        print(f'  This customer likely has some positive signals (e.g., recent web activity or moderate')
        print(f'  frequency) that masked the churn risk. The model was fooled by mixed signals.')
        print(f'  BUSINESS IMPACT: This customer churned without intervention. Lost revenue.')
    else:
        print(f'  WHY FALSE ALARM: Probability ({prob:.3f}) exceeds threshold ({SELECTED_THRESHOLD:.2f}).')
        print(f'  This customer had some risk indicators but ultimately stayed. Possible explanation:')
        print(f'  they may have received a competitor offer but decided to stay, or the model')
        print(f'  over-weighted a single risk factor (e.g., high recency) without capturing loyalty.')
        print(f'  BUSINESS IMPACT: Wasted ~₹30-40 campaign cost. Acceptable.')

## 10. Save Model & Export Metrics

In [ ]:
# ── Final evaluation on TEST set ──────────────────────────────────────────────
xgb_probs_test = xgb.predict_proba(X_test)[:, 1]
xgb_preds_test = (xgb_probs_test >= SELECTED_THRESHOLD).astype(int)

test_cm = confusion_matrix(y_test, xgb_preds_test)
tn, fp, fn, tp = test_cm.ravel()

test_metrics = {
    'model': 'XGBoost',
    'threshold': float(SELECTED_THRESHOLD),
    'test_accuracy': float((xgb_preds_test == y_test).mean()),
    'test_precision': float(tp / (tp + fp)) if (tp + fp) > 0 else 0,
    'test_recall': float(tp / (tp + fn)) if (tp + fn) > 0 else 0,
    'test_f1': float(f1_score(y_test, xgb_preds_test)),
    'test_roc_auc': float(roc_auc_score(y_test, xgb_probs_test)),
    'test_pr_auc': float(average_precision_score(y_test, xgb_probs_test)),
    'confusion_matrix': {'TN': int(tn), 'FP': int(fp), 'FN': int(fn), 'TP': int(tp)},
    'baseline_roc_auc': float(roc_auc_score(y_val, lr_probs_val)),
    'note': 'Threshold chosen to favour recall >= 70% for churn cost asymmetry'
}

print('=== FINAL TEST SET RESULTS ===')
print(classification_report(y_test, xgb_preds_test, target_names=['No Churn','Churned']))
print(f'ROC-AUC: {test_metrics["test_roc_auc"]:.4f}')
print(f'PR-AUC:  {test_metrics["test_pr_auc"]:.4f}')

In [ ]:
# ── Save model and metrics ────────────────────────────────────────────────────
joblib.dump(xgb, 'model.pkl')
print('Model saved → model.pkl')

# Also save the scaler and feature list for Part 4
joblib.dump(scaler, 'scaler.pkl')
joblib.dump(feature_cols_encoded, 'feature_cols.pkl')
print('Scaler saved → scaler.pkl')
print('Feature list saved → feature_cols.pkl')

with open('metrics.json', 'w') as f:
    json.dump(test_metrics, f, indent=2)
print('Metrics saved → metrics.json')

print('\nmetrics.json contents:')
print(json.dumps(test_metrics, indent=2))